# Explicit Small CNN for CIFAR-10

Small CNN classifier for CIFAR-10 (32x32x3 images, 10 classes).
Every convolution, ReLU, and pooling operation is a named layer
(no `nn.Sequential`), and the forward pass prints the `[B, C, H, W]`
shape after each individual operation. Purpose: build the habit of
treating tensor shapes as first-class before moving on to U-Net/ViT/
DiT architectures where shape bookkeeping gets much harder.

**Architecture** (`ExplicitCIFAR10CNN`, defined in
`code/cifar10_explicit_cnn.py`):

```
input                       [B,  3, 32, 32]
conv1 3x3 pad1 (3->16)   -> [B, 16, 32, 32]
relu1                    -> [B, 16, 32, 32]
pool1 2x2                -> [B, 16, 16, 16]
conv2 3x3 pad1 (16->32)  -> [B, 32, 16, 16]
relu2                    -> [B, 32, 16, 16]
pool2 2x2                -> [B, 32,  8,  8]
conv3 3x3 pad1 (32->64)  -> [B, 64,  8,  8]
relu3                    -> [B, 64,  8,  8]
pool3 2x2                -> [B, 64,  4,  4]
flatten                  -> [B, 1024]
fc1 (1024->128)          -> [B,  128]
relu4                    -> [B,  128]
fc2 (128->10)            -> [B,   10]   (logits)
```

Core code lives in `code/cifar10_explicit_cnn.py` (version-controlled),
this notebook is a thin orchestration layer: GPU check, repo pull,
build model, print shape trace + parameter count, download real
CIFAR-10 (cached in Google Drive so it is downloaded only once),
visualize the first 10 conv1 feature maps on a real image, train,
plot loss/accuracy curves, and show 5 sample predictions
(input / true label / predicted label) as a qualitative check.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Clone/pull repository

Model code lives in `code/cifar10_explicit_cnn.py`, not duplicated in
this notebook. Replace `REPO_URL` with this repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))

## 3. Build model, print shape trace and parameter count

One verbose forward pass on a small synthetic batch, purely to print
the `[B,C,H,W]` shape after every conv/ReLU/pool/linear op. Not part
of training.

In [ ]:
from cifar10_explicit_cnn import ExplicitCIFAR10CNN, count_trainable_parameters

torch.manual_seed(0)

model = ExplicitCIFAR10CNN(num_classes=10).to(device)
print(f"trainable parameters: {count_trainable_parameters(model):,}")

sample_batch = torch.randn(4, 3, 32, 32, device=device)  # [B,3,32,32], synthetic
_ = model(sample_batch, verbose=True)

## 4. Download real CIFAR-10 (train/test split), cached in Drive

`torchvision.datasets.CIFAR10(train=True)` / `train=False` gives the
dataset's native train/test partition. No extra validation split is
invented here.

Colab's local disk (`/content/...`) is wiped whenever the runtime
recycles, so the dataset would otherwise be re-downloaded on every
fresh session. Instead we mount Google Drive and point
`DATA_ROOT` at a persistent Drive folder: `torchvision` only
downloads into it once and simply reads from it on every later run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/cifar10"

In [ ]:
from cifar10_explicit_cnn import get_cifar10_dataloaders

BATCH_SIZE = 128

train_loader, test_loader = get_cifar10_dataloaders(DATA_ROOT, BATCH_SIZE)
print(f"train batches = {len(train_loader)}, test batches = {len(test_loader)}")

## 5. Visualize first 10 conv1 feature maps (real image, untrained model)

`conv1` applied to one real CIFAR-10 image gives `[1, 16, 32, 32]`;
we plot the first 10 of the 16 output channels (raw conv output,
before ReLU).

In [ ]:
from cifar10_explicit_cnn import visualize_first_conv_feature_maps

real_image, real_label = next(iter(test_loader))
real_image = real_image[0].to(device)  # [3, 32, 32]

visualize_first_conv_feature_maps(model, real_image, num_maps=10)

## 6. Train

Explicit train/eval loops (`train_one_epoch`, `evaluate`), CrossEntropy
loss, Adam optimizer.

In [ ]:
import torch.nn as nn
from cifar10_explicit_cnn import train_one_epoch, evaluate

N_EPOCHS = 10
LR = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"epoch {epoch + 1:3d}/{N_EPOCHS}  "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
        f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}"
    )

## 7. Loss / accuracy curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["test_loss"], label="test")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["test_acc"], label="test")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()
axes[1].set_title("Accuracy")

fig.tight_layout()
plt.show()

## 8. Feature maps after training (compare against step 5)

In [ ]:
visualize_first_conv_feature_maps(model, real_image, num_maps=10)

## 9. Qualitative check: sample predictions

Per-project convention: always look at concrete examples, not only
scalar metrics. Show 5 real test images together with the
ground-truth label and the model's predicted label (green title =
correct, red = wrong).

In [ ]:
from cifar10_explicit_cnn import visualize_predictions

sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

visualize_predictions(model, sample_images, sample_labels, num_examples=5)